In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv('d:/Data/Github/SheldonC2005/Academics/Machine Learning/titanic.csv')

print("="*70)
print("DATA VALIDATION AND ERROR CHECKING")
print("="*70)
print(f"\nOriginal Dataset Shape: {df.shape}")
print(f"Total Records: {df.shape[0]}")
print(f"\nColumns: {list(df.columns)}")

print(f"\n--- CHECKING FOR DATA ERRORS ---")
print(f"Duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Dataset after removing duplicates: {df.shape}")

print(f"\n--- MISSING VALUES DETECTED ---")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"Total missing values: {df.isnull().sum().sum()}")

print(f"\n--- DATA DISTRIBUTION ---")
print(f"Survival Rate: {df['Survived'].mean()*100:.2f}%")
print(f"Survived: {df['Survived'].sum()}, Not Survived: {(df['Survived']==0).sum()}")

y = df['Survived'].copy()
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()

print(f"\n--- FEATURE SELECTION ---")
print(f"Features selected for training: {list(X.columns)}")
print(f"Excluded identifier columns: PassengerId, Name, Ticket, Cabin")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

age_median = X_train['Age'].median()
embarked_mode = X_train['Embarked'].mode()[0]
fare_median = X_train['Fare'].median()

print(f"\n--- DATA IMPUTATION (Using Training Set Statistics) ---")
print(f"Age missing values filled with median: {age_median:.2f}")
print(f"Embarked missing values filled with mode: {embarked_mode}")
print(f"Fare missing values filled with median: {fare_median:.2f}")

X_train = X_train.copy()
X_train['Age'].fillna(age_median, inplace=True)
X_train['Embarked'].fillna(embarked_mode, inplace=True)
X_train['Fare'].fillna(fare_median, inplace=True)
X_train['Sex'] = X_train['Sex'].map({'male': 1, 'female': 0})
X_train['Embarked'] = X_train['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})
X_train['FamilySize'] = X_train['SibSp'] + X_train['Parch'] + 1

X_test = X_test.copy()
X_test['Age'].fillna(age_median, inplace=True)
X_test['Embarked'].fillna(embarked_mode, inplace=True)
X_test['Fare'].fillna(fare_median, inplace=True)
X_test['Sex'] = X_test['Sex'].map({'male': 1, 'female': 0})
X_test['Embarked'] = X_test['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})
X_test['FamilySize'] = X_test['SibSp'] + X_test['Parch'] + 1

print(f"\n--- ENCODING APPLIED ---")
print(f"Sex: male=1, female=0")
print(f"Embarked: C=0, Q=1, S=2")
print(f"Feature Engineering: FamilySize = SibSp + Parch + 1")

print(f"\n--- FINAL DATASET ---")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Features in model: {list(X_train.columns)}")
print(f"Missing values after preprocessing: {X_train.isnull().sum().sum()}")
print("="*70)

NameError: name 'pd' is not defined

In [ ]:
n_estimators = 100
print("\n" + "="*70)
print("MODEL TRAINING WITH ENSEMBLE METHODS")
print("="*70)

models = {
    'AdaBoost': AdaBoostClassifier(n_estimators=n_estimators, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=n_estimators, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=n_estimators, random_state=42, eval_metric='logloss'),
    'Random Forest': RandomForestClassifier(n_estimators=n_estimators, random_state=42)
}

results = {}
for name, model in models.items():
    print(f"\n{'='*70}")
    print(f"Training {name}")
    print(f"{'='*70}")
    print(f"Number of estimators (trees/weak learners): {n_estimators}")
    
    model.fit(X_train, y_train)
    
    print(f"Training completed: {model.n_estimators} iterations executed")
    
    if hasattr(model, 'estimators_'):
        print(f"Total weak learners created: {len(model.estimators_)}")
    
    if name in ['AdaBoost', 'Gradient Boosting']:
        print(f"\nBoosting Process:")
        print(f"- Each iteration corrects errors from previous iteration")
        print(f"- Sequential learning: Tree {n_estimators} fixes errors from Tree 1 to {n_estimators-1}")
        
    if name == 'Gradient Boosting' and hasattr(model, 'train_score_'):
        print(f"- Training score progression tracked across {len(model.train_score_)} iterations")
        print(f"- Initial training score: {model.train_score_[0]:.4f}")
        print(f"- Final training score: {model.train_score_[-1]:.4f}")
        print(f"- Error reduction: {(model.train_score_[-1] - model.train_score_[0]):.4f}")
    
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results[name] = {'accuracy': acc, 'predictions': pred, 'model': model}
    
    print(f"\nTest Set Performance:")
    print(f"Accuracy: {acc*100:.2f}%")
    print(f"\nClassification Report:")
    print(classification_report(y_test, pred, target_names=['Not Survived', 'Survived']))
    print(f"Confusion Matrix:")
    print(confusion_matrix(y_test, pred))

In [ ]:
comparison = pd.DataFrame([(k, v['accuracy']) for k, v in results.items()], 
                         columns=['Model', 'Accuracy']).sort_values('Accuracy', ascending=False)

print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(comparison.to_string(index=False))

best = comparison.iloc[0]
boosting = comparison[comparison['Model'].isin(['AdaBoost', 'Gradient Boosting', 'XGBoost'])]
rf_acc = comparison[comparison['Model'] == 'Random Forest']['Accuracy'].values[0]
best_boost_acc = boosting.iloc[0]['Accuracy']

print(f"\nBest Overall Model: {best['Model']} - {best['Accuracy']*100:.2f}%")
print(f"Best Boosting Model: {boosting.iloc[0]['Model']} - {best_boost_acc*100:.2f}%")
print(f"Performance Difference (Boosting vs Random Forest): {(best_boost_acc - rf_acc)*100:+.2f}%")

print(f"\n--- ERROR CORRECTION IN ENSEMBLE MODELS ---")
print(f"Boosting Models (AdaBoost, Gradient Boosting, XGBoost):")
print(f"  - Sequential learning across {n_estimators} iterations")
print(f"  - Each tree corrects misclassifications from previous trees")
print(f"  - Iterative error reduction strategy")
print(f"\nRandom Forest:")
print(f"  - Parallel learning with {n_estimators} independent trees")
print(f"  - Voting mechanism for final prediction")
print(f"  - Bagging approach reduces variance")

plt.figure(figsize=(10, 5))
plt.bar(comparison['Model'], comparison['Accuracy'], color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
plt.ylabel('Accuracy')
plt.title(f'Model Accuracy Comparison (n_estimators={n_estimators})')
plt.ylim([0.7, 0.9])
for i, row in comparison.iterrows():
    plt.text(i, row['Accuracy'], f"{row['Accuracy']*100:.2f}%", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print("="*70)